<a href="https://colab.research.google.com/github/sparsetrace/DMRG/blob/main/DMRG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os, sys, subprocess as sbp
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
try:
    import dmrg
except:
    sbp.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/sparsetrace/DMRG.git"])
    import dmrg

In [8]:
import os
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoImageProcessor, AutoModelForImageClassification

from dmrg import DMRG
from dmrg import DiffusionBlock

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HF_HUB_TOKEN")

# 1) Load processor + base model from the HF repo/subfolder
processor = AutoImageProcessor.from_pretrained(
    "jcandane/vit",
    subfolder="vit",
    trust_remote_code=False,
    token=HF_TOKEN,
)

model = AutoModelForImageClassification.from_pretrained(
    "jcandane/vit",
    subfolder="vit",
    trust_remote_code=False,
    token=HF_TOKEN,
)

# 2) Build a train loader
ds = load_dataset("ILSVRC/imagenet-1k", split="train", streaming=True, token=True)

def collate_fn(examples):
    images = []
    labels = []
    for ex in examples:
        img = ex["image"]
        if hasattr(img, "convert"):
            img = img.convert("RGB")
        images.append(img)
        labels.append(int(ex["label"]))

    batch = processor(images=images, return_tensors="pt")
    batch["labels"] = __import__("torch").tensor(labels, dtype=__import__("torch").long)
    return batch

train_loader = DataLoader(
    ds,
    batch_size=32,
    collate_fn=collate_fn,
)

# 3) Initialize DMRG
dmrg = DMRG(
    model=model,
    train_loader=train_loader,
    teacher="self",      # frozen copy of the original model
    hf_token=HF_TOKEN,
)

# 4a) Smoke test: first window only
dmrg.run_first_window(
    new_block=DiffusionBlock,
    block_kwargs={
        "channel_heads": 8,
        "dropout": 0.0,
    },
    steps_per_window=100,
    output_dir="./dmrg_ckpts",
    repo_id="jcandane/vit",
    hub_path="dmrg_test_first_window",
)

# 4b) Full run
# dmrg.run(
#     new_block=DiffusionBlock,
#     block_kwargs={
#         "channel_heads": 8,
#         "dropout": 0.0,
#     },
#     sweeps=1,
#     steps_per_window=500,
#     output_dir="./dmrg_ckpts",
#     repo_id="jcandane/vit",
#     hub_path="dmrg_full_run",
# )

OSError: jcandane/vit is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [7]:
from dmrg import DMRG